# Project 007: K-Nearest Neighbors (KNN) Spatial Geometry Pipeline

**Category:** Machine Learning Basic Projects  
**Domain:** Foundational Supervised Learning & Non-Parametric Spatial Geometry  
**Dataset:** Wisconsin Breast Cancer Diagnostic Dataset (569 samples, 30 features)  

---

### 📖 Theoretical Foundation

K-Nearest Neighbors (KNN) is an instance-based, non-parametric, lazy-learning algorithm that makes predictions based on the geometric proximity of training instances in a $D$-dimensional feature space.

$$
d_p(x, x') = \left( \sum_{i=1}^D |x_i - x'_i|^p \right)^{\frac{1}{p}}
$$

- $p = 1$: **Manhattan Distance ($L_1$)**
- $p = 2$: **Euclidean Distance ($L_2$)**
- $p \to \infty$: **Chebyshev Distance ($L_\infty$)**

#### ⚖️ The Bias-Variance Trade-Off in KNN:
- **$K = 1$**: Low bias, high variance (overfitting to local sample noise).
- **Optimal $K \approx \sqrt{N}$**: Balanced generalization error.
- **$K \to N$**: High bias, low variance (underfitting, always predicting majority class).

---

### 🎯 Notebook Workflow
1. **Environment & Imports**: Load scikit-learn, numpy, pandas, and project modules.
2. **Dataset Loading**: Examine the authentic Wisconsin Breast Cancer diagnostic dataset.
3. **Feature Scaling Analysis**: Demonstrate the critical necessity of feature standardization.
4. **K-Hyperparameter Optimization**: Run 5-fold cross-validation across $K \in [1, 25]$.
5. **Distance Metric Benchmark**: Compare Euclidean, Manhattan, Chebyshev, and Cosine.
6. **Champion Model Evaluation**: Assess Holdout Accuracy, Precision, Recall, F1, and Confusion Matrix.
7. **Neighbor Trace Inference**: Inspect the exact distances and classes of the $K$ closest neighbors.
8. **2D Spatial Decision Boundary**: Visualize non-linear manifold partitioning.

In [ ]:
# 1. Environment Setup & Imports
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from main import (
    load_dataset,
    train_knn_classifier,
    evaluate_knn,
    find_optimal_k,
    benchmark_distance_metrics
)
from dataset import generate_2d_manifold

print("Modules imported successfully.")

### 2. Dataset Loading & Inspection
We load the authentic Wisconsin Breast Cancer diagnostic benchmark dataset (569 patient biopsies across 30 nuclear morphology attributes).

In [ ]:
df, feature_names, target_col, target_names = load_dataset("breast_cancer")

print(f"Dataset Shape: {df.shape} (569 patient samples, 30 morphological features)")
print(f"Target Classes: {target_names}")
print(f"Class Distribution:\n{df['target_label'].value_counts()}\n")

# Inspect first 5 rows and summary statistics
display(df[feature_names[:6]].describe().round(2))
df.head(4)

### 3. Stratified Partitioning & Feature Scaling Sensitivity
Because KNN relies on geometric distances, features with large variance (e.g. `mean area` $\approx 650$) will completely drown out small-scale features (e.g. `mean smoothness` $\approx 0.09$) unless normalized.

In [ ]:
X = df[feature_names].values
y = df[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Unscaled Baseline
raw_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
raw_knn.fit(X_train, y_train)
raw_acc = accuracy_score(y_test, raw_knn.predict(X_test))

# Standardized with StandardScaler (Z-Score)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
scaled_knn.fit(X_train_scaled, y_train)
scaled_acc = accuracy_score(y_test, scaled_knn.predict(X_test_scaled))

print(f"Raw Unscaled Features Accuracy:  {raw_acc * 100:.2f}%")
print(f"StandardScaler Scaled Accuracy:  {scaled_acc * 100:.2f}% (+{(scaled_acc - raw_acc)*100:.2f}% performance boost)")

### 4. K-Hyperparameter Optimization via 5-Fold Cross-Validation
Evaluate cross-validation performance across $K \in [1, 21]$ to identify the optimal bias-variance equilibrium.

In [ ]:
best_k, best_cv, k_records = find_optimal_k(X_train_scaled, y_train, k_range=range(1, 23, 2))
k_df = pd.DataFrame(k_records)

print(f">>> Optimal K: {best_k} with {best_cv * 100:.2f}% 5-fold CV Accuracy\n")
k_df

### 5. Distance Metric Benchmarking
Evaluate holdout test accuracy across Euclidean ($L_2$), Manhattan ($L_1$), Chebyshev ($L_\infty$), Minkowski ($p=3$), and Cosine distance.

In [ ]:
metric_df = benchmark_distance_metrics(X_train_scaled, y_train, X_test_scaled, y_test, k=best_k)
metric_df

### 6. Champion Model Training & Holdout Diagnostics
Train the champion pipeline with optimal $K$ and Euclidean distance on standardized data.

In [ ]:
champion_model = train_knn_classifier(X_train_scaled, y_train, n_neighbors=best_k, metric='euclidean')
eval_res = evaluate_knn(champion_model, X_test_scaled, y_test, target_names=target_names)

print("=== CHAMPION MODEL HOLDOUT EVALUATION ===")
print(f"Accuracy:  {eval_res['accuracy'] * 100:.2f}%")
print(f"Precision: {eval_res['precision'] * 100:.2f}%")
print(f"Recall:    {eval_res['recall'] * 100:.2f}%")
print(f"F1-Score:  {eval_res['f1'] * 100:.2f}%\n")

print("Classification Report:")
print(eval_res['classification_report'])

print("Confusion Matrix:")
print(eval_res['confusion_matrix'])

### 7. Neighbor Trace & Inference Telemetry
For an unseen patient instance, examine the exact distances and ground truth labels of its $K$ closest neighbors in feature space.

In [ ]:
sample_idx = 0
sample_features = X_test_scaled[sample_idx:sample_idx+1]
true_label = target_names[y_test[sample_idx]]
pred_label = target_names[champion_model.predict(sample_features)[0]]
pred_proba = champion_model.predict_proba(sample_features)[0]

distances, indices = champion_model.kneighbors(sample_features, n_neighbors=best_k)

print(f"True Diagnostic Class:      {true_label}")
print(f"Predicted Diagnostic Class: {pred_label} (Confidence: {pred_proba[champion_model.predict(sample_features)[0]]*100:.1f}%)\n")
print(f"Exact {best_k} Nearest Training Neighbors:")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"  #{rank}: Distance = {dist:.3f} | Neighbor True Label: {target_names[y_train[idx]]}")

### 8. 2D Non-Linear Manifold & Voronoi Boundary Demonstration
Demonstrate KNN's flexible non-parametric decision boundaries on non-linearly separable Two Moons dataset.

In [ ]:
df_2d, feats_2d, target_2d, labels_2d = generate_2d_manifold("Two Moons", n_samples=300, noise=0.20)
knn_moons = KNeighborsClassifier(n_neighbors=7, metric='euclidean')
knn_moons.fit(df_2d[feats_2d].values, df_2d[target_2d].values)

train_acc = accuracy_score(df_2d[target_2d].values, knn_moons.predict(df_2d[feats_2d].values))
print(f"Two Moons Manifold - KNN (K=7) Accuracy: {train_acc * 100:.2f}%")
print("KNN readily accommodates highly non-linear curved geometries without kernel tricks!")